<a href="https://colab.research.google.com/github/thorntonjb/All-in-One-Gait/blob/main/colab-notebook/All_in_One_Gait_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# All-in-One-Gait on Google Colab

End-to-end gait recognition demo: **pedestrian tracking** (ByteTrack / YOLOX) → **silhouette segmentation** (PP-HumanSeg) → **gait recognition** (OpenGait GaitBase).

You give it one *gallery* video and one or more *probe* videos. Each person in a probe video is matched to a person in the gallery, and the result is drawn back onto the probe video (same bounding-box colour = same identity).

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or any GPU). The pipeline runs on CPU too, but tracking is ~50× slower.

Then run the cells top to bottom (`Runtime → Run all` works).

## 1. Check the GPU

In [1]:
!nvidia-smi || echo "No GPU detected - the demo will run on CPU (slow)."

Wed Oct 28 04:42:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code

The repository includes the five demo videos (~280 MB), so the clone takes a minute.

In [2]:
REPO_URL = "https://github.com/thorntonjb/All-in-One-Gait.git"  #@param {type:"string"}

import os
if not os.path.isdir("/content/All-in-One-Gait"):
    !git clone --depth 1 {REPO_URL} /content/All-in-One-Gait
%cd /content/All-in-One-Gait/OpenGait

Cloning into '/content/All-in-One-Gait'...
remote: Enumerating objects: 1537, done.
remote: Counting objects: 100% (1537/1537), done.
remote: Compressing objects: 100% (1445/1445), done.
remote: Total 1537 (delta 73), reused 1494 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (1537/1537), 311.88 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Updating files: 100% (1430/1430), done.
/content/All-in-One-Gait/OpenGait


## 3. Install dependencies

Colab already ships PyTorch, OpenCV, NumPy, SciPy, etc. Do **not** install the repo's `requirements.txt` here: it pins `torch==1.12` and `paddlepaddle==2.6.2`, neither of which has wheels for Colab's current Python. Only the missing packages are installed. PaddlePaddle 3.x still runs the old-format segmentation model.

In [3]:
!pip install -q "paddlepaddle>=3,<4" loguru lap filterpy thop kornia tabulate prettytable pycocotools tensorboard

## 4. Download the model checkpoints

| Model | Source |
|---|---|
| Gait recognition (GaitBase, GREW) | OpenGait GitHub release |
| Tracking (ByteTrack `bytetrack_x_mot17`) | Google Drive |
| Segmentation (PP-HumanSeg v2 mobile) | PaddleSeg |

If the Google Drive download fails (Drive sometimes rate-limits popular files), download [bytetrack_x_mot17.pth.tar](https://drive.google.com/file/d/1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5/view) yourself and upload it to `OpenGait/demo/checkpoints/bytetrack_model/` with the Files panel on the left, then re-run this cell.

In [4]:
%cd /content/All-in-One-Gait/OpenGait/demo/checkpoints

# Gait model
!mkdir -p gait_model
if not os.path.exists("gait_model/GaitBase_DA-180000.pt"):
    !wget -q --show-progress -O gait_model/pretrained_grew_gaitbase.zip https://github.com/ShiqiYu/OpenGait/releases/download/v2.0/pretrained_grew_gaitbase.zip
    !unzip -o -j -q gait_model/pretrained_grew_gaitbase.zip -d gait_model && rm gait_model/pretrained_grew_gaitbase.zip

# Tracking model
BYTETRACK = "bytetrack_model/bytetrack_x_mot17.pth.tar"
if not os.path.exists(BYTETRACK) or os.path.getsize(BYTETRACK) < 100_000_000:
    !pip install -q --upgrade gdown
    !gdown -O {BYTETRACK} "https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5"

# Segmentation model
SEG = "seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax"
if not os.path.exists(f"{SEG}/model.pdmodel"):
    !mkdir -p seg_model
    !wget -q --show-progress -O seg_model/seg.zip https://paddleseg.bj.bcebos.com/dygraph/pp_humanseg_v2/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip
    !unzip -o -q seg_model/seg.zip -d seg_model && rm -rf seg_model/seg.zip seg_model/__MACOSX

%cd /content/All-in-One-Gait/OpenGait

for path, min_mb in [("demo/checkpoints/gait_model/GaitBase_DA-180000.pt", 300),
                     ("demo/checkpoints/" + BYTETRACK, 300),
                     ("demo/checkpoints/" + SEG + "/model.pdiparams", 1)]:
    ok = os.path.exists(path) and os.path.getsize(path) > min_mb * 1e6
    print(("OK      " if ok else "MISSING ") + path)
    assert ok, f"{path} is missing or incomplete - see the note above this cell."

/content/All-in-One-Gait/OpenGait/demo/checkpoints
gait_model/pretrain 100%[===================>] 314.29M  9.75MB/s    in 30s     
Downloading...
From (original): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
From (redirected): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5&confirm=t&uuid=d316f3d0-d625-47cd-ba6f-35a8f0124e54
To: /content/All-in-One-Gait/OpenGait/demo/checkpoints/bytetrack_model/bytetrack_x_mot17.pth.tar
100% 793M/793M [00:10<00:00, 72.6MB/s]
seg_model/seg.zip   100%[===================>]  26.74M  22.3MB/s    in 1.2s    
/content/All-in-One-Gait/OpenGait
OK      demo/checkpoints/gait_model/GaitBase_DA-180000.pt
OK      demo/checkpoints/bytetrack_model/bytetrack_x_mot17.pth.tar
OK      demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax/model.pdiparams


## 5. Patch the code for current Colab

The repo was written for PyTorch 1.12 / NumPy 1.x. These small, idempotent patches make it run on Colab's PyTorch 2.x / NumPy 2 / Python 3.12+:

- `cython_bbox` doesn't build against NumPy 2 → replaced with an equivalent pure-NumPy IoU.
- `np.float` / `np.int` / `np.bool` were removed from NumPy → replaced with the builtins.
- The segmentation predictor was being rebuilt for every cropped frame → built once and reused.
- Tracks that never received a silhouette no longer crash the result-writing step.
- PyTorch ≥ 2.6 defaults `torch.load(weights_only=True)`, which rejects these older checkpoints → `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1`.

In [5]:
import os, re
from pathlib import Path

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
ROOT = Path("/content/All-in-One-Gait/OpenGait")

NUMPY_IOU = """try:
    from cython_bbox import bbox_overlaps as bbox_ious
except ImportError:
    def bbox_ious(boxes, query_boxes):
        boxes = np.asarray(boxes, dtype=float)
        query_boxes = np.asarray(query_boxes, dtype=float)
        area_b = (boxes[:, 2] - boxes[:, 0] + 1) * (boxes[:, 3] - boxes[:, 1] + 1)
        area_q = (query_boxes[:, 2] - query_boxes[:, 0] + 1) * (query_boxes[:, 3] - query_boxes[:, 1] + 1)
        iw = np.minimum(boxes[:, None, 2], query_boxes[None, :, 2]) - np.maximum(boxes[:, None, 0], query_boxes[None, :, 0]) + 1
        ih = np.minimum(boxes[:, None, 3], query_boxes[None, :, 3]) - np.maximum(boxes[:, None, 1], query_boxes[None, :, 1]) + 1
        inter = np.clip(iw, 0, None) * np.clip(ih, 0, None)
        union = area_b[:, None] + area_q[None, :] - inter
        return np.where(inter > 0, inter / union, 0.0)
"""


def patch(rel_path, fn):
    path = ROOT / rel_path
    src = path.read_text()
    new = fn(src)
    if new != src:
        path.write_text(new)
        print("patched", rel_path)


for rel in ["demo/libs/tracker/matching.py", "demo/libs/yolox/tracker/matching.py"]:
    patch(rel, lambda s: s if "except ImportError" in s else
          s.replace("from cython_bbox import bbox_overlaps as bbox_ious\n", NUMPY_IOU, 1))

for path in (ROOT / "demo/libs").rglob("*.py"):
    if "paddleseg" not in path.parts:
        patch(path.relative_to(ROOT), lambda s: re.sub(r"\bnp\.(float|int|bool)\b", r"\1", s))

patch("demo/libs/paddle/seg_demo.py", lambda s: s.replace(
    "def seg_image(img, config, save_name, savesil_path):\n    predictor = Predictor_opengait(config)\n",
    "_PREDICTORS = {}\n\n\ndef seg_image(img, config, save_name, savesil_path):\n"
    "    if config not in _PREDICTORS:\n        _PREDICTORS[config] = Predictor_opengait(config)\n"
    "    predictor = _PREDICTORS[config]\n", 1))

patch("demo/libs/track.py", lambda s: s.replace(
    "                    tid = pgdict[pid]\n",
    "                    if pid not in pgdict:\n                        continue\n"
    "                    tid = pgdict[pid]\n", 1) if "if pid not in pgdict" not in s else s)

print("patches applied")

patched demo/libs/tracker/matching.py
patched demo/libs/yolox/tracker/matching.py
patched demo/libs/tracker/matching.py
patched demo/libs/yolox/motdt_tracker/motdt_tracker.py
patched demo/libs/yolox/motdt_tracker/matching.py
patched demo/libs/yolox/motdt_tracker/reid_model.py
patched demo/libs/yolox/tracker/byte_tracker.py
patched demo/libs/yolox/tracker/matching.py
patched demo/libs/yolox/deepsort_tracker/detection.py
patched demo/libs/yolox/deepsort_tracker/deepsort.py
patched demo/libs/paddle/seg_demo.py
patched demo/libs/track.py
patches applied


## 6. (Optional) Use your own videos

By default the demo uses the bundled `gallery.mp4` and `probe1-4.mp4`. To use your own, run the upload cell, then set `GALLERY` / `PROBES` in step 7 to the uploaded file names.

Tips: one or a few people walking, full body visible, ideally a side view. Shorter clips (5–15 s) run faster.

In [ ]:
import os, re
from google.colab import files

INPUT_DIR = "/content/All-in-One-Gait/OpenGait/demo/output/InputVideos"
uploaded = files.upload()
for name, data in uploaded.items():
    # Track IDs are built as "<video>-<id>" and split on "-", so keep names hyphen-free.
    safe = re.sub(r"[^A-Za-z0-9_.]", "_", name)
    with open(os.path.join(INPUT_DIR, safe), "wb") as f:
        f.write(data)
    print("saved", safe)
print("Available videos:", sorted(os.listdir(INPUT_DIR)))

## 7. Run the pipeline

For each video: track people → segment silhouettes → extract gait features. Then every probe is compared against the gallery and an annotated result video is written.

With a T4 GPU the bundled demo (five ~8 s videos) takes roughly 10–15 minutes; most of it is segmentation, which runs on CPU.

In [6]:
GALLERY = "gallery.mp4"  #@param {type:"string"}
PROBES = "probe1.mp4, probe2.mp4, probe3.mp4, probe4.mp4"  #@param {type:"string"}

import os, sys, shutil, time
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.chdir("/content/All-in-One-Gait/OpenGait")
if "./demo/libs" not in sys.path:
    sys.path.insert(0, os.path.abspath("demo/libs"))

from track import track, writeresult
from segment import seg
from recognise import extract_sil, compare

input_dir = "./demo/output/InputVideos"
sil_root = "./demo/output/GaitSilhouette/"
feat_root = "./demo/output/GaitFeatures/"
out_dir = os.path.join("./demo/output/OutputVideos", time.strftime("%Y_%m_%d_%H_%M_%S"))

probes = [p.strip() for p in PROBES.split(",") if p.strip()]
videos = [GALLERY] + probes
for v in videos:
    assert os.path.exists(os.path.join(input_dir, v)), f"{v} not found in {input_dir}"

feats = {}
for v in videos:
    path = os.path.join(input_dir, v)
    name = os.path.splitext(v)[0]
    # The repo ships silhouettes from earlier runs; start fresh so they don't mix with new ones.
    shutil.rmtree(os.path.join(sil_root, name), ignore_errors=True)
    shutil.rmtree(os.path.join(feat_root, name), ignore_errors=True)
    print(f"\n=== {v}: tracking ===")
    track_result = track(path, out_dir)
    print(f"=== {v}: segmentation ===")
    sils = seg(path, track_result, sil_root)
    print(f"=== {v}: gait features ===")
    feats[v] = extract_sil(sils, feat_root)

results = {}
for p in probes:
    print(f"\n=== {p} vs {GALLERY} ===")
    results[p] = compare(feats[p], feats[GALLERY])
    writeresult(results[p], os.path.join(input_dir, p), out_dir)

print("\nResults written to", os.path.abspath(out_dir))

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/content/All-in-One-Gait/OpenGait/demo/libs/paddle/paddleseg/models/losses/rmi_loss.py:76: SyntaxWarning: invalid escape sequence '\i'
  logits (Tensor): Shape is [N, C, H, W], logits at each prediction (between -\infty and +\infty).
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
2026-10-28 04:50:58.626 | INFO     | track:loadckpt:53 - Model Summary: Params: 99.00M, Gflops: 793.21
2026-10-28 04:50:58.62


=== gallery.mp4: tracking ===
video save_path is ./demo/output/OutputVideos/2026_10_28_04_51_03/gallery.mp4


100%|██████████| 235/235 [00:41<00:00,  5.68it/s]
2026-10-28 04:51:45.145 | INFO     | track:track:154 - save results to ./demo/output/OutputVideos/2026_10_28_04_51_03/gallery.txt


=== gallery.mp4: segmentation ===


100%|██████████| 235/235 [00:21<00:00, 10.78it/s]
2026-10-28 04:52:07.016 | INFO     | recognise:extract_sil:95 - begin extracting


...


=== probe3.mp4 vs gallery.mp4 ===
================= probe - gallery ===================
{'probe3-001': 'gallery-002', 'probe3-002': 'gallery-003', 'probe3-004': 'gallery-001', 'probe3-003': 'gallery-004'}
video save_path is ./demo/output/OutputVideos/2026_10_28_04_51_03/G-gallery_P-probe3.mp4


100%|██████████| 212/212 [00:39<00:00,  5.34it/s]
2026-10-28 04:58:40.041 | INFO     | recognise:compare:110 - begin recognising
2026-10-28 04:58:40.046 | INFO     | recognise:compare:112 - recognise Done



=== probe4.mp4 vs gallery.mp4 ===
================= probe - gallery ===================
{'probe4-001': 'gallery-004', 'probe4-002': 'gallery-003', 'probe4-004': 'gallery-004', 'probe4-003': 'gallery-004'}
video save_path is ./demo/output/OutputVideos/2026_10_28_04_51_03/G-gallery_P-probe4.mp4


100%|██████████| 161/161 [00:34<00:00,  4.67it/s]


Results written to /content/All-in-One-Gait/OpenGait/demo/output/OutputVideos/2026_10_28_04_51_03


## 8. Results

Each probe person `probeX-NNN` is assigned the closest gallery person `gallery-MMM`. In the result videos the box colour and label show the matched gallery identity.

In [7]:
for p, mapping in results.items():
    print(p)
    for probe_id, gallery_id in mapping.items():
        print(f"   {probe_id}  ->  {gallery_id}")

probe1.mp4
   probe1-001  ->  gallery-004
   probe1-002  ->  gallery-003
   probe1-004  ->  gallery-001
   probe1-003  ->  gallery-002
probe2.mp4
   probe2-001  ->  gallery-002
   probe2-002  ->  gallery-004
   probe2-004  ->  gallery-001
   probe2-003  ->  gallery-003
probe3.mp4
   probe3-001  ->  gallery-002
   probe3-002  ->  gallery-003
   probe3-004  ->  gallery-001
   probe3-003  ->  gallery-004
probe4.mp4
   probe4-001  ->  gallery-004
   probe4-002  ->  gallery-003
   probe4-004  ->  gallery-004
   probe4-003  ->  gallery-004


In [8]:
import glob, subprocess
from base64 import b64encode
from IPython.display import HTML, display

def show(path, width=360):
    # OpenCV writes mp4v, which browsers can't play; re-encode a small H.264 preview.
    preview = path[:-4] + "_preview.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vf", "scale=480:-2",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "28", preview], check=True)
    data = b64encode(open(preview, "rb").read()).decode()
    display(HTML(f'<p><b>{os.path.basename(path)}</b></p>'
                 f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

show(os.path.join(out_dir, GALLERY))
for p in probes:
    show(os.path.join(out_dir, f"G-{os.path.splitext(GALLERY)[0]}_P-{p}"))

## 9. Download everything

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/gait_results", "zip", out_dir)
files.download(archive)